# AttGAN — Facial Attribute Editing with GANs
**Dataset:** CelebA (Kaggle) · **Framework:** PyTorch · **Platform:** Google Colab  
> He et al. (2019) — *AttGAN: Facial Attribute Editing by Only Changing What You Want*

---
## Before you run anything

### Step A — Set runtime to GPU
`Runtime → Change runtime type → T4 GPU → Save`

### Step B — Download CelebA from Kaggle
1. Go to https://www.kaggle.com/datasets/jessicali9530/celeba-dataset
2. Click **Download** — you get a ZIP containing several files
3. Extract it. You need **exactly these 3 items**:
   - `img_align_celeba.zip` → unzip this too → gives you a folder `img_align_celeba/` with 202,599 `.jpg` files
   - `list_attr_celeba.csv`
   - `list_eval_partition.csv`

### Step C — Upload to Google Drive
1. Open [Google Drive](https://drive.google.com)
2. Create this exact folder structure:
```
My Drive/
  celeba_data/
      img_align_celeba/      <- paste the 202,599 jpg files here
          000001.jpg
          000002.jpg
          ...
      list_attr_celeba.csv   <- upload directly here
      list_eval_partition.csv <- upload directly here
```
3. Uploading 200k images takes time — use **Google Drive desktop app** or drag-drop the entire `img_align_celeba` folder

### Step D — Run the cells in order
Cell 3 mounts your Drive. Cell 4 points the code to `celeba_data/`.  
**Run this notebook once per experiment** (change `EXPERIMENT` in Cell 5).

---
| Experiment | λ_rec | λ_cls_G | Hypothesis |
|---|---|---|---|
| `exp1_baseline` | 100 | 1 | Paper defaults — balanced |
| `exp2_high_rec` | 200 | 1 | Stronger identity, softer edits |
| `exp3_strong_attr` | 50 | 5 | Sharper edits, less identity |

---
## Cell 1 — Clone repo & install

In [ ]:
# If repo already cloned from a previous session, skip the clone line
!git clone https://github.com/YOUR_USERNAME/GAN_Project_DL.git
%cd GAN_Project_DL
!pip install -q -r requirements.txt

import torch, sys
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('NO GPU — go to Runtime > Change Runtime Type > GPU')

---
## Cell 2 — Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU found. Go to Runtime > Change Runtime Type > T4 GPU'
print(f'GPU ready: {torch.cuda.get_device_name(0)}')

---
## Cell 3 — Mount Google Drive & locate dataset
This mounts your Drive and verifies the three required files are present.  
**If you uploaded to a different path**, change `CELEBA_DRIVE_PATH` below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ── Change this if you uploaded to a different folder name ───────────
CELEBA_DRIVE_PATH = '/content/drive/MyDrive/celeba_data'
# ────────────────────────────────────────────────────────────────────

celeba_dir = Path(CELEBA_DRIVE_PATH)

# Verify all required files exist
required = [
    celeba_dir / 'img_align_celeba',
    celeba_dir / 'list_attr_celeba.csv',
    celeba_dir / 'list_eval_partition.csv',
]
all_ok = True
for p in required:
    status = 'OK' if p.exists() else 'MISSING'
    if status == 'MISSING': all_ok = False
    print(f'  [{status}]  {p}')

if not all_ok:
    raise FileNotFoundError(
        'One or more files are missing. '
        'Re-read Step C in the header above and re-upload.'
    )

# Count images as a sanity check
n_imgs = len(list((celeba_dir / 'img_align_celeba').glob('*.jpg')))
print(f'\nImages found : {n_imgs:,}  (expected 202,599)')
print('Dataset ready!')

---
## Cell 4 — Select experiment & build config
**Change `EXPERIMENT` here to switch between the three configurations.**  
Run the full notebook once per experiment, then run Cell 12 to compare.

In [ ]:
import importlib, torch

# ── CHANGE THIS to switch experiments ───────────────────────────────
EXPERIMENT = 'exp1_baseline'
# Options:
#   'exp1_baseline'    -> lambda_rec=100  lambda_cls_g=1   (paper defaults)
#   'exp2_high_rec'    -> lambda_rec=200  lambda_cls_g=1   (stronger identity)
#   'exp3_strong_attr' -> lambda_rec=50   lambda_cls_g=5   (sharper edits)
# ────────────────────────────────────────────────────────────────────

_EXP_MAP = {
    'exp1_baseline':    ('experiments.exp1_baseline', 'Exp1Config'),
    'exp2_high_rec':    ('experiments.exp2_high_rec', 'Exp2Config'),
    'exp3_strong_attr': ('experiments.exp3_low_rec',  'Exp3Config'),
}
mod_path, cls_name = _EXP_MAP[EXPERIMENT]
cfg = getattr(importlib.import_module(mod_path), cls_name)()

# Point config to the Drive dataset
cfg.CELEBA_DIR = celeba_dir

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Experiment   : {cfg.EXPERIMENT_NAME}')
print(f'Device       : {device}')
print(f'lambda_rec   : {cfg.LAMBDA_REC}')
print(f'lambda_cls_D : {cfg.LAMBDA_CLS_D}')
print(f'lambda_cls_G : {cfg.LAMBDA_CLS_G}')
print(f'Epochs       : {cfg.N_EPOCHS}')
print(f'Batch size   : {cfg.BATCH_SIZE}')
print(f'Results dir  : {cfg.RESULTS_DIR}')
print(f'CelebA dir   : {cfg.CELEBA_DIR}')
if hasattr(cfg, 'DESCRIPTION'):
    print(f'Description  : {cfg.DESCRIPTION}')

---
## Cell 5 — Load dataset
Uses the Kaggle CSV files directly — no Google Drive download quota issues.  

**Splits used** (identical to torchvision defaults):
| Split | Partition code | Images |
|---|---|---|
| train | 0 | 162,770 |
| valid | 1 | 19,867 |
| test  | 2 | 19,962 |

In [ ]:
from src.dataset import get_loaders

train_loader, test_loader = get_loaders(cfg)

# Sanity check one batch
imgs, attrs = next(iter(train_loader))
print(f'Image batch : {imgs.shape}   (B, C, H, W)')
print(f'Attr  batch : {attrs.shape}  values in {set(attrs.unique().tolist())}')
print(f'Pixel range : [{imgs.min():.2f}, {imgs.max():.2f}]  (should be close to -1, +1)')

---
## Cell 6 — Build models
```
image (3x128x128)
   |-> Encoder (5x Conv stride-2)  -> z (512x4x4)
                                         |
                       target_attrs -----+  (tiled spatially, concat)
                                         |
                                    Generator (5x ConvTranspose) -> fake (3x128x128)
                                                                         |
                                    Discriminator <----------------------+
                                         |-> adv_head  (LSGAN real/fake)
                                         |-> cls_head  (13 attribute logits BCE)
```

In [ ]:
from src.models import build_models

enc, gen, dis = build_models(cfg, device)

---
## Cell 7 — Train

| Loss | Function | Weight | Purpose |
|---|---|---|---|
| L_adv | MSELoss (LSGAN) | — | Realism |
| L_cls | BCEWithLogitsLoss | lambda_cls_D / lambda_cls_G | Correct attributes |
| L_rec | L1Loss | lambda_rec | Identity preservation |

Samples + checkpoints saved to `results/<experiment>/` every 5 epochs.

In [ ]:
from src.trainer import Trainer

trainer = Trainer(enc, gen, dis, train_loader, test_loader, cfg, device)

# To resume from a saved checkpoint, uncomment:
# resume = f'checkpoints/{cfg.EXPERIMENT_NAME}/ckpt_epoch010.pt'
# g_losses, d_losses = trainer.train(resume_path=resume)

g_losses, d_losses = trainer.train()

---
## Cell 8 — Loss curves

In [ ]:
from src.utils import plot_losses
plot_losses(g_losses, d_losses, cfg)

---
## Cell 9 — Attribute manipulation demo
Each row = one test image.  
Each column = one attribute toggled from its current value (all other attributes unchanged).  
This is the main qualitative result of AttGAN.

In [ ]:
from src.utils import attribute_demo
attribute_demo(enc, gen, test_loader, cfg, n_imgs=4)

---
## Cell 10 — Attribute accuracy & reconstruction quality

In [ ]:
from src.utils import evaluate_attribute_accuracy, evaluate_reconstruction

acc = evaluate_attribute_accuracy(enc, gen, dis, test_loader, cfg)
rec = evaluate_reconstruction(enc, gen, test_loader, cfg)

print(f'\nAttribute accuracy (avg) : {acc:.1f}%')
print(f'Reconstruction L1        : {rec:.4f}')

---
## Cell 11 — FID & DACID metrics
**FID** (Frechet Inception Distance): Frechet distance between Inception-v3 feature
distributions of real vs generated images. Standard GAN quality metric. Lower = better.

**DACID** (Dany Aissa & Clara's Image Distance): L2 distance between *mean* Inception
feature vectors. Lighter and faster than FID, good for quick experiment comparisons. Lower = better.

> Takes ~5 extra minutes. Skip with `cfg.COMPUTE_METRICS = False` to iterate faster.

In [ ]:
# cfg.COMPUTE_METRICS = False  # uncomment to skip

if cfg.COMPUTE_METRICS:
    from src.metrics import compute_metrics
    scores = compute_metrics(enc, gen, test_loader, cfg, device)
    print(f"FID   : {scores['fid']}")
    print(f"DACID : {scores['dacid']}")
else:
    scores = {}
    print('Metrics skipped.')

---
## Cell 12 — Save metrics.json
Saves everything to `results/<experiment>/metrics.json`.  
**Run this at the end of every experiment** — the export cell reads these files.

In [ ]:
import json

payload = {
    'experiment':   cfg.EXPERIMENT_NAME,
    'model':        'AttGAN',
    'lambda_rec':   cfg.LAMBDA_REC,
    'lambda_cls_d': cfg.LAMBDA_CLS_D,
    'lambda_cls_g': cfg.LAMBDA_CLS_G,
    'fid':          scores.get('fid'),
    'dacid':        scores.get('dacid'),
    'attr_acc':     round(float(acc), 2) if 'acc' in dir() else None,
    'rec_l1':       round(float(rec), 4) if 'rec' in dir() else None,
    'g_losses':     g_losses,
    'd_losses':     d_losses,
}

out = cfg.RESULTS_DIR / 'metrics.json'
with open(out, 'w') as f:
    json.dump(payload, f, indent=2)

print(f'Saved -> {out}')
print(f'\nSummary — {cfg.EXPERIMENT_NAME}:')
for k in ['lambda_rec', 'lambda_cls_g', 'fid', 'dacid', 'attr_acc', 'rec_l1']:
    print(f'  {k:<16}: {payload[k]}')

---
## Cell 13 — Export & compare all experiments
**Run after completing ALL THREE experiments.**  
Produces `results/comparison_table.csv`, `comparison_metrics.png`, `comparison_losses.png`.

In [ ]:
# Import and call export_results main directly
import importlib.util, sys
spec = importlib.util.spec_from_file_location('export_results', 'export_results.py')
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.main()

---
## Troubleshooting

**`FileNotFoundError: Missing CelebA files`**  
Re-check Step C in the header. The folder structure must be exactly:
```
MyDrive/celeba_data/
    img_align_celeba/   <- folder with .jpg files, NOT another zip
    list_attr_celeba.csv
    list_eval_partition.csv
```

**`Drive not mounted`**  
Re-run Cell 3. If it hangs, go to `Runtime -> Disconnect and delete runtime` and start again.

**`CUDA out of memory`**  
Reduce batch size: add `cfg.BATCH_SIZE = 16` right after Cell 4.